In [4]:
import inspect
from langchain.agents.middleware import SummarizationMiddleware

print(inspect.signature(SummarizationMiddleware))
print(inspect.getsource(SummarizationMiddleware._normalize_trigger))

(model: str | langchain_core.language_models.chat_models.BaseChatModel, *, trigger: tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int] | langchain.agents.middleware.summarization.TriggerClause | list[tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int] | langchain.agents.middleware.summarization.TriggerClause] | None = None, keep: tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int] = ('messages', 20), token_counter: collections.abc.Callable[[collections.abc.Iterable[langchain_core.messages.base.BaseMessage | list[str] | tuple[str, str | list[str | dict[str, typing.Any]]] | str | dict[str, typing.Any]]], int] = <function count_tokens_approximately at 0x110811120>, summary_prompt: str = '<role>\nContext Extraction Assistant\n</role>\n\n<primary_objective>\nYour sole objective in this 

In [6]:
import inspect
from langchain.agents import create_agent

print(inspect.signature(create_agent))

(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None, transformers: 'Sequence[TransformerFactory] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]'


In [18]:

from langchain.agents import create_agent
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
load_dotenv()

system_message=SystemMessage("You are a helpful AI Research Assistant. Always provides facts and examples in response")



class AIResearch(TypedDict):
    """AI Research Assitant answer Details"""
    answer: Annotated[str, ...,'Answer of the AI Question']
    sources: Annotated[str, ...,'Give all sources reference links'] 
    confidence: Annotated[int, ...,"Score the answer's confidence scale based on 10"]
    tool_used: Annotated[str, ...,"Give the list of all tools used"]
@tool
def web_search(query:str)->str:
    """Search for information in detailed"""
    return f"Results for :{query}"


@tool
def calculator(equation:str)->str:
    """Perform any calculation for user"""
    return f"Results for :{equation}"

agent=create_agent(
    model='gpt-4o-mini',
    tools=[web_search,calculator],
    system_prompt=system_message,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("messages",10),
            keep=("messages",4)
        )
    ],
    response_format=AIResearch
)

# 5. Configure conversation memory
config = {
    "configurable": {
        "thread_id": "test-12"
    }
}

for chunk, metadata in agent.stream({"messages":[HumanMessage(content="Explain how RAG works")]},config,stream_mode="messages"):
        if chunk.content:
             
            print(chunk.content, end="", flush=True)

print()

Results for :Retrieval-Augmented Generation (RAG) model explanation{"answer":"Retrieval-Augmented Generation (RAG) is a model architecture that combines two key components: a retrieval system and a generative model. It is particularly useful in natural language processing tasks where providing accurate and relevant information is critical. Here’s how RAG works:\n\n1. **Retrieval Mechanism**:  The first part of RAG involves retrieving relevant documents or passages from a large knowledge base. This is done using a retrieval model (often based on dense vector representations or BM25, a traditional information retrieval model). The goal is to find a small number of relevant texts based on a given query.\n\n2. **Query Processing**: A query, which could be a question or a prompt, is processed to identify related documents. This can involve encoding the query into a vector in the same space as the encoded documents in the knowledge base.\n\n3. **Document Retrieval**: The retrieval system fet

In [ ]:
print(agent.get_input_jsonschema())f

{'$defs': {'AIMessage': {'additionalProperties': True, 'description': 'Message from an AI.\n\nAn `AIMessage` is returned from a chat model as a response to a prompt.\n\nThis message represents the output of the model and consists of both\nthe raw output as returned by the model and standardized fields\n(e.g., tool calls, usage metadata) added by the LangChain framework.', 'properties': {'content': {'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'type': 'string'}, {'additionalProperties': True, 'type': 'object'}]}, 'type': 'array'}], 'title': 'Content'}, 'additional_kwargs': {'additionalProperties': True, 'title': 'Additional Kwargs', 'type': 'object'}, 'response_metadata': {'additionalProperties': True, 'title': 'Response Metadata', 'type': 'object'}, 'type': {'const': 'ai', 'default': 'ai', 'title': 'Type', 'type': 'string'}, 'name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Name'}, 'id': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'defaul